# CRUD dinámico de pedidos desde Excel

Este notebook usa **`BaseDatosProyecto.xlsx`**, hoja **`G. Pedidos`**, como fuente de datos.

Incluye:

- carga del Excel en un `DataFrame` de pandas;
- columna calculada **Precio Total** = `Precio unidad × Cantidad × (1 - Descuento)`;
- formulario CRUD con `ipywidgets`, similar al formulario por filas de Excel;
- navegación registro anterior/siguiente;
- creación, modificación, búsqueda y eliminación;
- guardado en un archivo de trabajo separado: **`BaseDatosProyecto_CRUD.xlsx`**;
- dashboard dinámico por **país** y **ciudad**;
- gráficos con `matplotlib`.

> El archivo original no se sobrescribe.


In [7]:
%pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Si te falta alguna librería, ejecuta esta celda una sola vez:
# %pip install pandas openpyxl matplotlib ipywidgets

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True


In [2]:
# ==========================
# 1. CARGA Y NORMALIZACIÓN
# ==========================

ARCHIVO_ORIGINAL = Path("BaseDatosProyecto.xlsx")
ARCHIVO_TRABAJO = Path("BaseDatosProyecto_CRUD.xlsx")
HOJA = "G. Pedidos"

if not ARCHIVO_ORIGINAL.exists():
    raise FileNotFoundError(
        f"No se encuentra {ARCHIVO_ORIGINAL}. "
        "Coloca este notebook en la misma carpeta que el Excel."
    )

df = pd.read_excel(ARCHIVO_ORIGINAL, sheet_name=HOJA)

# Normalizar nombres de columnas procedentes de Excel
df.columns = (
    df.columns.astype(str)
      .str.replace("\n", " ", regex=False)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

renombres = {
    "Ciudad destino": "Ciudad destino",
    "Precio unidad": "Precio unidad",
}
df = df.rename(columns=renombres)

# Fechas
for col in ["Fecha Pedido", "Fecha Entrega"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Tipos numéricos
for col in ["Precio unidad", "Cantidad", "Descuento"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# ID interno estable para el CRUD
if "ID" not in df.columns:
    df.insert(0, "ID", range(1, len(df) + 1))

def recalcular_precio_total(dataframe):
    dataframe["Precio Total"] = (
        dataframe["Precio unidad"].astype(float)
        * dataframe["Cantidad"].astype(float)
        * (1 - dataframe["Descuento"].astype(float))
    ).round(2)
    return dataframe

df = recalcular_precio_total(df)

print(f"Registros cargados: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
display(df.head())


Registros cargados: 2,155
Columnas: ['ID', 'Cliente', 'Artículo', 'Fecha Pedido', 'Fecha Entrega', 'País Destino', 'Ciudad destino', 'Precio unidad', 'Cantidad', 'Descuento', 'Precio Total']


,ID,Cliente,Artículo,Fecha Pedido,Fecha Entrega,País Destino,Ciudad destino,Precio unidad,Cantidad,Descuento,Precio Total
0,1,Alfreds Futterkiste,Arenque salado,2019-08-25,2019-09-22,Alemania,Berlin,18.615939,20,0.25,279.24
1,2,Alfreds Futterkiste,Caracoles de Borgoña,2020-04-09,2020-05-07,Alemania,Berlin,20.555064,40,0.05,781.09
2,3,Alfreds Futterkiste,Col fermentada Rössle,2019-08-25,2019-09-22,Alemania,Berlin,70.740567,15,0.25,795.83
3,4,Alfreds Futterkiste,Col fermentada Rössle,2020-03-16,2020-04-27,Alemania,Berlin,70.740567,2,0.00,141.48
4,5,Alfreds Futterkiste,Crema de queso Fløtemys,2020-04-09,2020-05-07,Alemania,Berlin,33.353486,20,0.00,667.07


In [3]:
# ==========================
# 2. FUNCIONES DE PERSISTENCIA
# ==========================

def guardar_en_excel():
    global df
    df = recalcular_precio_total(df.copy())
    columnas_salida = [
        "Cliente", "Artículo", "Fecha Pedido", "Fecha Entrega",
        "País Destino", "Ciudad destino", "Precio unidad",
        "Cantidad", "Descuento", "Precio Total"
    ]
    df[columnas_salida].to_excel(
        ARCHIVO_TRABAJO,
        sheet_name=HOJA,
        index=False
    )
    return ARCHIVO_TRABAJO

def siguiente_id():
    return 1 if df.empty else int(df["ID"].max()) + 1

print("Archivo de trabajo:", ARCHIVO_TRABAJO)


Archivo de trabajo: BaseDatosProyecto_CRUD.xlsx


## 3. Formulario CRUD

El formulario trabaja **registro por registro**, igual que el formulario clásico de Excel.

- **◀ Anterior / Siguiente ▶**: navega por las filas.
- **Nuevo**: prepara un registro vacío.
- **Guardar**: crea o actualiza.
- **Eliminar**: elimina el registro actual.
- **Buscar**: localiza coincidencias por ID, cliente, artículo, país o ciudad.
- El **Precio Total** se recalcula inmediatamente cuando cambias precio, cantidad o descuento.


In [4]:
# ==========================
# 3. CRUD INTERACTIVO
# ==========================

posicion_actual = 0
modo_nuevo = False
resultados_busqueda = []
posicion_busqueda = -1

w_id = widgets.IntText(description="ID:", disabled=True)
w_cliente = widgets.Text(description="Cliente:", layout=widgets.Layout(width="650px"))
w_articulo = widgets.Text(description="Artículo:", layout=widgets.Layout(width="650px"))
w_fecha_pedido = widgets.DatePicker(description="Fecha pedido:")
w_fecha_entrega = widgets.DatePicker(description="Fecha entrega:")
w_pais = widgets.Text(description="País:", layout=widgets.Layout(width="500px"))
w_ciudad = widgets.Text(description="Ciudad:", layout=widgets.Layout(width="500px"))
w_precio = widgets.FloatText(description="Precio unidad:", step=0.01)
w_cantidad = widgets.IntText(description="Cantidad:", value=1)
w_descuento = widgets.BoundedFloatText(
    description="Descuento:",
    min=0,
    max=1,
    step=0.01,
    value=0
)
w_total = widgets.FloatText(description="Precio Total:", disabled=True)

w_buscar = widgets.Text(
    description="Buscar:",
    placeholder="ID, cliente, artículo, país o ciudad",
    layout=widgets.Layout(width="650px")
)

btn_anterior = widgets.Button(description="◀ Anterior")
btn_siguiente = widgets.Button(description="Siguiente ▶")
btn_nuevo = widgets.Button(description="Nuevo", button_style="info")
btn_guardar = widgets.Button(description="Guardar", button_style="success")
btn_eliminar = widgets.Button(description="Eliminar", button_style="danger")
btn_buscar = widgets.Button(description="Buscar")
btn_sig_busqueda = widgets.Button(description="Siguiente coincidencia")
btn_exportar = widgets.Button(description="Guardar Excel", button_style="warning")

estado = widgets.HTML()
salida_form = widgets.Output()

def precio_total_formulario():
    precio = float(w_precio.value or 0)
    cantidad = int(w_cantidad.value or 0)
    descuento = float(w_descuento.value or 0)
    w_total.value = round(precio * cantidad * (1 - descuento), 2)

for control in [w_precio, w_cantidad, w_descuento]:
    control.observe(lambda cambio: precio_total_formulario(), names="value")

def fecha_python(valor):
    if pd.isna(valor):
        return None
    return pd.Timestamp(valor).date()

def mostrar_registro(pos):
    global posicion_actual, modo_nuevo
    if df.empty:
        return
    pos = max(0, min(pos, len(df) - 1))
    posicion_actual = pos
    modo_nuevo = False
    fila = df.iloc[pos]

    w_id.value = int(fila["ID"])
    w_cliente.value = str(fila["Cliente"])
    w_articulo.value = str(fila["Artículo"])
    w_fecha_pedido.value = fecha_python(fila["Fecha Pedido"])
    w_fecha_entrega.value = fecha_python(fila["Fecha Entrega"])
    w_pais.value = str(fila["País Destino"])
    w_ciudad.value = str(fila["Ciudad destino"])
    w_precio.value = float(fila["Precio unidad"])
    w_cantidad.value = int(fila["Cantidad"])
    w_descuento.value = float(fila["Descuento"])
    precio_total_formulario()

    estado.value = (
        f"<b>Registro {pos + 1} de {len(df)}</b> — "
        f"ID {int(fila['ID'])}"
    )

def limpiar_formulario():
    global modo_nuevo
    modo_nuevo = True
    w_id.value = siguiente_id()
    w_cliente.value = ""
    w_articulo.value = ""
    w_fecha_pedido.value = pd.Timestamp.today().date()
    w_fecha_entrega.value = pd.Timestamp.today().date()
    w_pais.value = ""
    w_ciudad.value = ""
    w_precio.value = 0
    w_cantidad.value = 1
    w_descuento.value = 0
    precio_total_formulario()
    estado.value = "<b>Nuevo registro</b>"

def valores_formulario():
    return {
        "ID": int(w_id.value),
        "Cliente": w_cliente.value.strip(),
        "Artículo": w_articulo.value.strip(),
        "Fecha Pedido": pd.Timestamp(w_fecha_pedido.value) if w_fecha_pedido.value else pd.NaT,
        "Fecha Entrega": pd.Timestamp(w_fecha_entrega.value) if w_fecha_entrega.value else pd.NaT,
        "País Destino": w_pais.value.strip(),
        "Ciudad destino": w_ciudad.value.strip(),
        "Precio unidad": float(w_precio.value),
        "Cantidad": int(w_cantidad.value),
        "Descuento": float(w_descuento.value),
        "Precio Total": float(w_total.value),
    }

def validar_formulario():
    if not w_cliente.value.strip():
        return False, "El cliente es obligatorio."
    if not w_articulo.value.strip():
        return False, "El artículo es obligatorio."
    if w_precio.value < 0:
        return False, "El precio no puede ser negativo."
    if w_cantidad.value < 0:
        return False, "La cantidad no puede ser negativa."
    if not 0 <= w_descuento.value <= 1:
        return False, "El descuento debe estar entre 0 y 1."
    return True, ""

def accion_anterior(_):
    mostrar_registro(posicion_actual - 1)

def accion_siguiente(_):
    mostrar_registro(posicion_actual + 1)

def accion_nuevo(_):
    limpiar_formulario()

def accion_guardar(_):
    global df, posicion_actual, modo_nuevo
    valido, mensaje = validar_formulario()
    if not valido:
        estado.value = f"<span style='color:#b00020'><b>Error:</b> {mensaje}</span>"
        return

    registro = valores_formulario()

    if modo_nuevo:
        df = pd.concat([df, pd.DataFrame([registro])], ignore_index=True)
        posicion_actual = len(df) - 1
        modo_nuevo = False
        mensaje = "Registro creado."
    else:
        idx = df.index[df["ID"] == registro["ID"]]
        if len(idx) == 0:
            estado.value = "<span style='color:#b00020'>No se encontró el ID.</span>"
            return
        for columna, valor in registro.items():
            df.loc[idx[0], columna] = valor
        posicion_actual = int(df.index.get_loc(idx[0]))
        mensaje = "Registro actualizado."

    df = recalcular_precio_total(df)
    guardar_en_excel()
    mostrar_registro(posicion_actual)
    estado.value += f" <span style='color:green'>✓ {mensaje}</span>"
    actualizar_dashboard()

def accion_eliminar(_):
    global df, posicion_actual
    if df.empty or modo_nuevo:
        estado.value = "<span style='color:#b00020'>No hay un registro guardado seleccionado.</span>"
        return

    id_actual = int(w_id.value)
    df = df[df["ID"] != id_actual].reset_index(drop=True)
    guardar_en_excel()

    if df.empty:
        limpiar_formulario()
        estado.value = "<span style='color:green'>✓ Registro eliminado. La tabla quedó vacía.</span>"
    else:
        posicion_actual = min(posicion_actual, len(df) - 1)
        mostrar_registro(posicion_actual)
        estado.value += " <span style='color:green'>✓ Registro eliminado.</span>"

    actualizar_dashboard()

def buscar_indices(texto):
    texto = texto.strip()
    if not texto:
        return []

    mask = pd.Series(False, index=df.index)

    if texto.isdigit():
        mask |= df["ID"].astype(str).eq(texto)

    columnas = ["Cliente", "Artículo", "País Destino", "Ciudad destino"]
    for col in columnas:
        mask |= df[col].astype(str).str.contains(texto, case=False, na=False, regex=False)

    return df.index[mask].tolist()

def accion_buscar(_):
    global resultados_busqueda, posicion_busqueda
    resultados_busqueda = buscar_indices(w_buscar.value)
    posicion_busqueda = 0

    if not resultados_busqueda:
        estado.value = "<span style='color:#b00020'>Sin coincidencias.</span>"
        return

    mostrar_registro(resultados_busqueda[0])
    estado.value += f" — coincidencia 1 de {len(resultados_busqueda)}"

def accion_sig_busqueda(_):
    global posicion_busqueda
    if not resultados_busqueda:
        accion_buscar(None)
        return

    posicion_busqueda = (posicion_busqueda + 1) % len(resultados_busqueda)
    mostrar_registro(resultados_busqueda[posicion_busqueda])
    estado.value += (
        f" — coincidencia {posicion_busqueda + 1} "
        f"de {len(resultados_busqueda)}"
    )

def accion_exportar(_):
    archivo = guardar_en_excel()
    estado.value = f"<span style='color:green'>✓ Guardado en <b>{archivo}</b></span>"

btn_anterior.on_click(accion_anterior)
btn_siguiente.on_click(accion_siguiente)
btn_nuevo.on_click(accion_nuevo)
btn_guardar.on_click(accion_guardar)
btn_eliminar.on_click(accion_eliminar)
btn_buscar.on_click(accion_buscar)
btn_sig_busqueda.on_click(accion_sig_busqueda)
btn_exportar.on_click(accion_exportar)

formulario = widgets.VBox([
    estado,
    widgets.HBox([btn_anterior, btn_siguiente, btn_nuevo, btn_guardar, btn_eliminar, btn_exportar]),
    widgets.HBox([w_buscar, btn_buscar, btn_sig_busqueda]),
    w_id,
    w_cliente,
    w_articulo,
    widgets.HBox([w_fecha_pedido, w_fecha_entrega]),
    widgets.HBox([w_pais, w_ciudad]),
    widgets.HBox([w_precio, w_cantidad, w_descuento, w_total]),
])

display(formulario)

if not df.empty:
    mostrar_registro(0)
else:
    limpiar_formulario()


## 4. Vista tabular dinámica

Esta celda permite filtrar rápidamente el `DataFrame` sin modificar los datos.


In [8]:
# ==========================
# 4. VISTA TABULAR DINÁMICA
# ==========================

# Normalizar nombres de columnas por seguridad
df.columns = (
    df.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Asegurar nombres esperados
df = df.rename(columns={
    "Ciudad destino": "Ciudad destino",
    "Precio unidad": "Precio unidad"
})

# Crear Precio Total si todavía no existe
if "Precio Total" not in df.columns:
    df["Precio Total"] = (
        df["Precio unidad"]
        * df["Cantidad"]
        * (1 - df["Descuento"])
    ).round(2)

filtro_tabla = widgets.Text(
    description="Filtrar:",
    placeholder="Cliente, artículo, país o ciudad",
    layout=widgets.Layout(width="700px")
)

limite_tabla = widgets.IntSlider(
    description="Filas:",
    min=5,
    max=100,
    step=5,
    value=20
)

salida_tabla = widgets.Output()

def actualizar_tabla(*_):
    texto = filtro_tabla.value.strip()
    vista = df.copy()

    columnas_busqueda = [
        "Cliente",
        "Artículo",
        "País Destino",
        "Ciudad destino"
    ]

    if texto:
        mask = pd.Series(False, index=vista.index)

        for col in columnas_busqueda:
            if col in vista.columns:
                mask = mask | vista[col].astype(str).str.contains(
                    texto,
                    case=False,
                    na=False,
                    regex=False
                )

        vista = vista[mask]

    with salida_tabla:
        clear_output(wait=True)

        columnas_formato = {}

        if "Precio unidad" in vista.columns:
            columnas_formato["Precio unidad"] = "{:.2f} €"

        if "Descuento" in vista.columns:
            columnas_formato["Descuento"] = "{:.0%}"

        if "Precio Total" in vista.columns:
            columnas_formato["Precio Total"] = "{:.2f} €"

        display(
            vista
            .head(limite_tabla.value)
            .style
            .format(columnas_formato)
        )

        print(f"Coincidencias: {len(vista):,}")

filtro_tabla.observe(
    actualizar_tabla,
    names="value"
)

limite_tabla.observe(
    actualizar_tabla,
    names="value"
)

display(
    widgets.HBox([
        filtro_tabla,
        limite_tabla
    ]),
    salida_tabla
)

actualizar_tabla()

Output()

## 5. Dashboard dinámico por país y ciudad

Los indicadores y gráficos se recalculan al cambiar los filtros.

Estadísticas incluidas:

- número de registros;
- unidades vendidas;
- ventas netas (`Precio Total`);
- precio medio por unidad;
- descuento medio;
- ventas por ciudad;
- artículos con mayor facturación.


In [9]:
# ==========================
# 5. DASHBOARD DINÁMICO
# ==========================

selector_pais = widgets.Dropdown(description="País:")
selector_ciudad = widgets.Dropdown(description="Ciudad:")
top_n = widgets.IntSlider(description="Top:", min=3, max=15, value=8)

kpis = widgets.HTML()
salida_dashboard = widgets.Output()

def refrescar_opciones():
    paises = ["Todos"] + sorted(
        x for x in df["País Destino"].dropna().astype(str).unique() if x
    )

    pais_actual = selector_pais.value
    selector_pais.options = paises
    if pais_actual in paises:
        selector_pais.value = pais_actual
    else:
        selector_pais.value = "Todos"

    base = df if selector_pais.value == "Todos" else df[df["País Destino"] == selector_pais.value]
    ciudades = ["Todas"] + sorted(
        x for x in base["Ciudad destino"].dropna().astype(str).unique() if x
    )

    ciudad_actual = selector_ciudad.value
    selector_ciudad.options = ciudades
    if ciudad_actual in ciudades:
        selector_ciudad.value = ciudad_actual
    else:
        selector_ciudad.value = "Todas"

def datos_filtrados():
    datos = df.copy()

    if selector_pais.value and selector_pais.value != "Todos":
        datos = datos[datos["País Destino"] == selector_pais.value]

    if selector_ciudad.value and selector_ciudad.value != "Todas":
        datos = datos[datos["Ciudad destino"] == selector_ciudad.value]

    return datos

def actualizar_dashboard(*_):
    if "selector_pais" not in globals():
        return

    refrescar_opciones()
    datos = datos_filtrados()

    registros = len(datos)
    unidades = int(datos["Cantidad"].sum()) if registros else 0
    ventas = float(datos["Precio Total"].sum()) if registros else 0
    precio_medio = float(datos["Precio unidad"].mean()) if registros else 0
    descuento_medio = float(datos["Descuento"].mean()) if registros else 0

    kpis.value = f'''
    <div style="display:flex;gap:14px;flex-wrap:wrap;margin:10px 0">
      <div style="padding:10px 16px;border:1px solid #ccc;border-radius:8px"><b>Registros</b><br>{registros:,}</div>
      <div style="padding:10px 16px;border:1px solid #ccc;border-radius:8px"><b>Unidades</b><br>{unidades:,}</div>
      <div style="padding:10px 16px;border:1px solid #ccc;border-radius:8px"><b>Ventas netas</b><br>{ventas:,.2f} €</div>
      <div style="padding:10px 16px;border:1px solid #ccc;border-radius:8px"><b>Precio medio</b><br>{precio_medio:,.2f} €</div>
      <div style="padding:10px 16px;border:1px solid #ccc;border-radius:8px"><b>Descuento medio</b><br>{descuento_medio:.1%}</div>
    </div>
    '''

    with salida_dashboard:
        clear_output(wait=True)

        if datos.empty:
            print("No hay datos para los filtros seleccionados.")
            return

        ventas_ciudad = (
            datos.groupby("Ciudad destino", dropna=False)["Precio Total"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(top_n.value)
                 .sort_values()
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        ventas_ciudad.plot(kind="barh", ax=ax)
        ax.set_title("Ventas netas por ciudad")
        ax.set_xlabel("Ventas (€)")
        ax.set_ylabel("Ciudad")
        plt.tight_layout()
        plt.show()

        ventas_pais = (
            datos.groupby("País Destino", dropna=False)["Precio Total"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(top_n.value)
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        ventas_pais.plot(kind="bar", ax=ax)
        ax.set_title("Ventas netas por país")
        ax.set_xlabel("País")
        ax.set_ylabel("Ventas (€)")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()

        articulos = (
            datos.groupby("Artículo")["Precio Total"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(top_n.value)
                 .sort_values()
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        articulos.plot(kind="barh", ax=ax)
        ax.set_title("Artículos con mayor facturación")
        ax.set_xlabel("Ventas (€)")
        ax.set_ylabel("Artículo")
        plt.tight_layout()
        plt.show()

        resumen = (
            datos.groupby(["País Destino", "Ciudad destino"], dropna=False)
                 .agg(
                     Registros=("ID", "count"),
                     Unidades=("Cantidad", "sum"),
                     Ventas=("Precio Total", "sum"),
                     Descuento_medio=("Descuento", "mean")
                 )
                 .sort_values("Ventas", ascending=False)
                 .head(20)
        )

        display(
            resumen.style.format({
                "Unidades": "{:,.0f}",
                "Ventas": "{:,.2f} €",
                "Descuento_medio": "{:.1%}"
            })
        )

def cambio_pais(cambio):
    if cambio["name"] == "value":
        refrescar_opciones()
        actualizar_dashboard()

selector_pais.observe(cambio_pais, names="value")
selector_ciudad.observe(actualizar_dashboard, names="value")
top_n.observe(actualizar_dashboard, names="value")

refrescar_opciones()

display(
    widgets.HBox([selector_pais, selector_ciudad, top_n]),
    kpis,
    salida_dashboard
)
actualizar_dashboard()


HTML(value='\n    <div style="display:flex;gap:14px;flex-wrap:wrap;margin:10px 0">\n      <div style="padding:…

Output()

## 6. Comprobaciones rápidas

Esta celda sirve para verificar que el precio total se mantiene coherente después de los cambios CRUD.


In [10]:
df = recalcular_precio_total(df)

comprobacion = (
    df["Precio Total"]
    - (
        df["Precio unidad"]
        * df["Cantidad"]
        * (1 - df["Descuento"])
      ).round(2)
).abs().max()

print("Diferencia máxima en Precio Total:", comprobacion)
print("Registros:", len(df))
print("Ventas netas:", f"{df['Precio Total'].sum():,.2f} €")


Diferencia máxima en Precio Total: 0.0
Registros: 2155
Ventas netas: 1,963,911.99 €
